In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Create Spark Session
spark = SparkSession.builder.appName("hospitalAnalysis").getOrCreate()

# Load Dataset
df = spark.read.csv("/Volumes/workspace/default/hospital/Hospital.csv",
                    header=True,
                    inferSchema=True)


print(df.columns)
df.printSchema()

['OrganisationID', 'OrganisationCode', 'OrganisationType', 'SubType', 'Sector', 'OrganisationStatus', 'IsPimsManaged', 'OrganisationName', 'Address1', 'Address2', 'Address3', 'City', 'County', 'Postcode', 'Latitude', 'Longitude', 'ParentODSCode', 'ParentName', 'Phone', 'Email', 'Website', 'Fax,,,']
root
 |-- OrganisationID: integer (nullable = true)
 |-- OrganisationCode: integer (nullable = true)
 |-- OrganisationType: string (nullable = true)
 |-- SubType: string (nullable = true)
 |-- Sector: string (nullable = true)
 |-- OrganisationStatus: string (nullable = true)
 |-- IsPimsManaged: string (nullable = true)
 |-- OrganisationName: boolean (nullable = true)
 |-- Address1: string (nullable = true)
 |-- Address2: string (nullable = true)
 |-- Address3: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- Postcode: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- ParentODSC

In [0]:
df.show()
for c in df.columns:
    new_col = c.strip().replace(" ", "").replace(",", "")
    df = df.withColumnRenamed(c, new_col)

# Remove Duplicates
df = df.dropDuplicates()

# Handle Missing Values
df = df.fillna({
    "OrganisationType":"Unknown",
    "SubType":"Unknown",
    "Sector":"Unknown",
    "OrganisationStatus":"Unknown",
    "City":"Unknown",
    "County":"Unknown",
    "ParentName":"Unknown",
    "Phone":"Not Available",
    "Email":"Not Available",
    "Website":"Not Available",
    "Fax":"Not Available"
})

# Convert Data Types
df = df.withColumn("OrganisationID",col("OrganisationID").cast("int"))
df = df.withColumn("Latitude",col("Latitude").cast("double"))
df = df.withColumn("Longitude",col("Longitude").cast("double"))

+--------------+----------------+----------------+--------+--------+------------------+-------------+----------------+--------------------+--------------------+------------------+--------------------+----------------+------------------+--------+------------------+--------------------+----------+--------------------+-------------+--------------------+--------------------+
|OrganisationID|OrganisationCode|OrganisationType| SubType|  Sector|OrganisationStatus|IsPimsManaged|OrganisationName|            Address1|            Address2|          Address3|                City|          County|          Postcode|Latitude|         Longitude|       ParentODSCode|ParentName|               Phone|        Email|             Website|              Fax,,,|
+--------------+----------------+----------------+--------+--------+------------------+-------------+----------------+--------------------+--------------------+------------------+--------------------+----------------+------------------+--------+-------

In [0]:
organisation_types = df.select("OrganisationType").distinct() \
.withColumn("organisation_type_id",monotonically_increasing_id())

df = df.join(organisation_types,"OrganisationType")

# Cities
cities = df.select("City").distinct() \
.withColumn("city_id",monotonically_increasing_id())

df = df.join(cities,"City")

# Parent Organisations
parents = df.select("ParentName").distinct() \
.withColumn("parent_id",monotonically_increasing_id())

df = df.join(parents,"ParentName")

# Hospital Table
hospitals = df.select(
    col("OrganisationID").alias("hospital_id"),
    "OrganisationCode",
    "OrganisationName",
    "organisation_type_id",
    "city_id",
    "parent_id",
    "OrganisationStatus",
    "Sector",
    "Latitude",
    "Longitude"
)

# Rename Dimension Columns

organisation_types = organisation_types.select(
    "organisation_type_id",
    col("OrganisationType").alias("organisation_type")
)

cities = cities.select(
    "city_id",
    col("City").alias("city_name")
)

parents = parents.select(
    "parent_id",
    col("ParentName").alias("parent_name")
)

In [0]:
hospitals.createOrReplaceTempView("hospitals")
organisation_types.createOrReplaceTempView("organisation_types")
cities.createOrReplaceTempView("cities")
parents.createOrReplaceTempView("parents")

In [0]:
query1 = spark.sql("""
SELECT h.OrganisationName,
       c.city_name
FROM hospitals h
JOIN cities c
ON h.city_id = c.city_id
""")


# 2. List all active hospitals

query2 = spark.sql("""
SELECT OrganisationName,
       OrganisationStatus
FROM hospitals
WHERE OrganisationStatus='Active'
""")
# 3. Count hospitals in each city

query3 = spark.sql("""
SELECT c.city_name,
COUNT(*) AS hospital_count
FROM hospitals h
JOIN cities c
ON h.city_id=c.city_id
GROUP BY c.city_name
ORDER BY hospital_count DESC
""")

# 4. Count hospitals by organisation type

query4 = spark.sql("""
SELECT o.organisation_type,
COUNT(*) AS total_hospitals
FROM hospitals h
JOIN organisation_types o
ON h.organisation_type_id=o.organisation_type_id
GROUP BY o.organisation_type
ORDER BY total_hospitals DESC
""")
# 5. Hospitals in Public Sector

query5 = spark.sql("""
SELECT OrganisationName,
Sector
FROM hospitals
WHERE Sector='Public'
""")

# 6. Count hospitals by parent organisation

query6 = spark.sql("""
SELECT p.parent_name,
COUNT(*) AS hospital_count
FROM hospitals h
JOIN parents p
ON h.parent_id=p.parent_id
GROUP BY p.parent_name
ORDER BY hospital_count DESC
""")
# 7. Cities having more than 10 hospitals

threshold = 10

query7 = spark.sql(f"""
SELECT c.city_name,
COUNT(*) AS hospital_count
FROM hospitals h
JOIN cities c
ON h.city_id=c.city_id
GROUP BY c.city_name
HAVING COUNT(*)>{threshold}
ORDER BY hospital_count DESC
""")
# 8. Hospitals with available website

query8 = spark.sql("""
SELECT Sector,
COUNT(*) AS total_hospitals
FROM hospitals
GROUP BY Sector
ORDER BY total_hospitals DESC
""")


# 9. Hospitals with available phone number

query9 = spark.sql("""
SELECT c.city_name,
COUNT(*) AS total_hospitals
FROM hospitals h
JOIN cities c
ON h.city_id = c.city_id
GROUP BY c.city_name
ORDER BY total_hospitals ASC
""")

# 10. Hospitals grouped by status

query10 = spark.sql("""
SELECT OrganisationStatus,
COUNT(*) AS total_hospitals
FROM hospitals
GROUP BY OrganisationStatus
ORDER BY total_hospitals DESC
""")


OrganisationName,city_name
true,Unknown
true,Unknown
true,Unknown
true,Unknown
true,Woodingdean
true,Unknown
true,Unknown
true,Unknown
true,Chelsfield
true,Unknown


OrganisationName,OrganisationStatus


city_name,hospital_count
Unknown,1064
Headington,7
Erdington,3
Canford Cliffs,2
Bebington,2
Knaphill,2
Edgbaston,2
Lewes Road,2
Liverpool Road,2
Moseley,2


organisation_type,total_hospitals
NMJ2F,1
RYYC7,1
TAJ83,1
RJR60,1
RTQHM,1
RYR16,1
RALRA,1
NMJ0Y,1
RWH01,1
RJL30,1


OrganisationName,Sector


parent_name,hospital_count
NT4,43
NT3,35
NVC,31
NT2,29
NMJ,24
RDY,20
RXT,18
NTN,18
RH5,15
RW1,14


city_name,hospital_count
Unknown,1064


Sector,total_hospitals
Hospital,961
UNKNOWN,244
Mental Health Hospital,6


city_name,total_hospitals
Market Drayton,1
Stockton On The Forest,1
Northfield,1
Old Wells Road,1
Benwick Road,1
Benham Hill,1
Thomas Drive,1
Norton,1
Brintons Terrace,1
Stepney Green,1


OrganisationStatus,total_hospitals
NHS Sector,743
Independent Sector,468
